# Churn Campaign ETL Demo Notebook

Self-contained Apache Spark notebook for an AI Data Platform demo.

What it does:
- Simulates churn-related campaign data
- Cleans and standardizes raw records
- Joins campaign activity with churn outcomes
- Builds dashboard-ready aggregates for reporting

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.0


## 1) Create synthetic source data

In [2]:
campaign_rows = [
    # customer_id, campaign_id, campaign_name, channel, send_date, opened, clicked, cost_usd, segment, region
    (101, "CAMP-001", "Winback Email A", "email", "2026-03-01", 1, 0, 0.12, "SMB", "NA"),
    (102, "CAMP-001", "Winback Email A", "email", "2026-03-01", 1, 1, 0.12, "Enterprise", "NA"),
    (103, "CAMP-001", "Winback Email A", "email", "2026-03-01", 0, 0, 0.12, "SMB", "EU"),
    (104, "CAMP-002", "Save Offer SMS", "sms",   "2026-03-02", 1, 1, 0.08, "Consumer", "NA"),
    (105, "CAMP-002", "Save Offer SMS", "sms",   "2026-03-02", 0, 0, 0.08, "Consumer", "APAC"),
    (106, "CAMP-003", "Loyalty Push",   "push",  "2026-03-03", 1, 0, 0.05, "SMB", "EU"),
    (107, "CAMP-003", "Loyalty Push",   "push",  "2026-03-03", 1, 1, 0.05, "Enterprise", "APAC"),
    (108, "CAMP-004", "Retention Call", "call",  "2026-03-04", 0, 0, 2.50, "Enterprise", "NA"),
    (109, "CAMP-004", "Retention Call", "call",  "2026-03-04", 1, 0, 2.50, "SMB", "EU"),
    (110, "CAMP-005", "Cross-sell Email", "email", "2026-03-05", 1, 1, 0.10, "Consumer", "NA"),
]

campaign_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("campaign_id", T.StringType(), False),
    T.StructField("campaign_name", T.StringType(), False),
    T.StructField("channel", T.StringType(), False),
    T.StructField("send_date", T.StringType(), False),
    T.StructField("opened", T.IntegerType(), False),
    T.StructField("clicked", T.IntegerType(), False),
    T.StructField("cost_usd", T.DoubleType(), False),
    T.StructField("segment", T.StringType(), False),
    T.StructField("region", T.StringType(), False),
])

raw_campaign_df = spark.createDataFrame(campaign_rows, schema=campaign_schema)

churn_rows = [
    # customer_id, churned_30d, arr_usd, tenure_months, last_login_days_ago
    (101, 1, 120.0, 3, 14),
    (102, 0, 800.0, 24, 2),
    (103, 1, 95.0, 2, 27),
    (104, 0, 50.0, 6, 1),
    (105, 1, 35.0, 4, 21),
    (106, 0, 150.0, 10, 5),
    (107, 0, 900.0, 30, 0),
    (108, 1, 1200.0, 18, 40),
    (109, 1, 75.0, 5, 18),
    (110, 0, 200.0, 12, 3),
]

churn_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("churned_30d", T.IntegerType(), False),
    T.StructField("arr_usd", T.DoubleType(), False),
    T.StructField("tenure_months", T.IntegerType(), False),
    T.StructField("last_login_days_ago", T.IntegerType(), False),
])

raw_churn_df = spark.createDataFrame(churn_rows, schema=churn_schema)

print("Campaign source")
raw_campaign_df.show(truncate=False)
print("Churn source")
raw_churn_df.show(truncate=False)

Campaign source


+-----------+-----------+----------------+-------+----------+------+-------+--------+----------+------+
|customer_id|campaign_id|campaign_name   |channel|send_date |opened|clicked|cost_usd|segment   |region|
+-----------+-----------+----------------+-------+----------+------+-------+--------+----------+------+
|101        |CAMP-001   |Winback Email A |email  |2026-03-01|1     |0      |0.12    |SMB       |NA    |
|102        |CAMP-001   |Winback Email A |email  |2026-03-01|1     |1      |0.12    |Enterprise|NA    |
|103        |CAMP-001   |Winback Email A |email  |2026-03-01|0     |0      |0.12    |SMB       |EU    |
|104        |CAMP-002   |Save Offer SMS  |sms    |2026-03-02|1     |1      |0.08    |Consumer  |NA    |
|105        |CAMP-002   |Save Offer SMS  |sms    |2026-03-02|0     |0      |0.08    |Consumer  |APAC  |
|106        |CAMP-003   |Loyalty Push    |push   |2026-03-03|1     |0      |0.05    |SMB       |EU    |
|107        |CAMP-003   |Loyalty Push    |push   |2026-03-03|1  

+-----------+-----------+-------+-------------+-------------------+
|customer_id|churned_30d|arr_usd|tenure_months|last_login_days_ago|
+-----------+-----------+-------+-------------+-------------------+
|101        |1          |120.0  |3            |14                 |
|102        |0          |800.0  |24           |2                  |
|103        |1          |95.0   |2            |27                 |
|104        |0          |50.0   |6            |1                  |
|105        |1          |35.0   |4            |21                 |
|106        |0          |150.0  |10           |5                  |
|107        |0          |900.0  |30           |0                  |
|108        |1          |1200.0 |18           |40                 |
|109        |1          |75.0   |5            |18                 |
|110        |0          |200.0  |12           |3                  |
+-----------+-----------+-------+-------------+-------------------+



## 2) Clean and standardize the campaign feed

In [3]:
campaign_df = (
    raw_campaign_df
    .withColumn("send_date", F.to_date("send_date"))
    .withColumn("opened", F.col("opened").cast("boolean").cast("int"))
    .withColumn("clicked", F.col("clicked").cast("boolean").cast("int"))
    .withColumn("channel", F.lower(F.trim("channel")))
    .withColumn("campaign_name", F.trim("campaign_name"))
    .withColumn("segment", F.initcap(F.lower(F.trim("segment"))))
    .withColumn("region", F.upper(F.trim("region")))
    .withColumn("is_engaged", F.when((F.col("opened") == 1) | (F.col("clicked") == 1), 1).otherwise(0))
    .withColumn("campaign_month", F.date_format("send_date", "yyyy-MM"))
)

churn_df = (
    raw_churn_df
    .withColumn("churned_30d", F.col("churned_30d").cast("int"))
    .withColumn("arr_usd", F.round("arr_usd", 2))
    .withColumn("tenure_band",
                F.when(F.col("tenure_months") < 6, "0-5")
                 .when(F.col("tenure_months") < 12, "6-11")
                 .when(F.col("tenure_months") < 24, "12-23")
                 .otherwise("24+"))
)

campaign_df.show(truncate=False)
churn_df.show(truncate=False)

+-----------+-----------+----------------+-------+----------+------+-------+--------+----------+------+----------+--------------+
|customer_id|campaign_id|campaign_name   |channel|send_date |opened|clicked|cost_usd|segment   |region|is_engaged|campaign_month|
+-----------+-----------+----------------+-------+----------+------+-------+--------+----------+------+----------+--------------+
|101        |CAMP-001   |Winback Email A |email  |2026-03-01|1     |0      |0.12    |Smb       |NA    |1         |2026-03       |
|102        |CAMP-001   |Winback Email A |email  |2026-03-01|1     |1      |0.12    |Enterprise|NA    |1         |2026-03       |
|103        |CAMP-001   |Winback Email A |email  |2026-03-01|0     |0      |0.12    |Smb       |EU    |0         |2026-03       |
|104        |CAMP-002   |Save Offer SMS  |sms    |2026-03-02|1     |1      |0.08    |Consumer  |NA    |1         |2026-03       |
|105        |CAMP-002   |Save Offer SMS  |sms    |2026-03-02|0     |0      |0.08    |Consu

+-----------+-----------+-------+-------------+-------------------+-----------+
|customer_id|churned_30d|arr_usd|tenure_months|last_login_days_ago|tenure_band|
+-----------+-----------+-------+-------------+-------------------+-----------+
|101        |1          |120.0  |3            |14                 |0-5        |
|102        |0          |800.0  |24           |2                  |24+        |
|103        |1          |95.0   |2            |27                 |0-5        |
|104        |0          |50.0   |6            |1                  |6-11       |
|105        |1          |35.0   |4            |21                 |0-5        |
|106        |0          |150.0  |10           |5                  |6-11       |
|107        |0          |900.0  |30           |0                  |24+        |
|108        |1          |1200.0 |18           |40                 |12-23      |
|109        |1          |75.0   |5            |18                 |0-5        |
|110        |0          |200.0  |12     

## 3) Join campaign and churn data

In [4]:
fact_df = (
    campaign_df.alias("c")
    .join(churn_df.alias("h"), on="customer_id", how="left")
    .withColumn("risk_flag", F.when((F.col("churned_30d") == 1) & (F.col("last_login_days_ago") > 14), 1).otherwise(0))
    .withColumn("touch_cost_usd", F.round(F.col("cost_usd"), 2))
    .select(
        "customer_id", "campaign_id", "campaign_name", "channel", "send_date",
        "campaign_month", "opened", "clicked", "is_engaged", "touch_cost_usd",
        "segment", "region", "churned_30d", "arr_usd", "tenure_months",
        "tenure_band", "last_login_days_ago", "risk_flag"
    )
)

fact_df.show(truncate=False)

+-----------+-----------+----------------+-------+----------+--------------+------+-------+----------+--------------+----------+------+-----------+-------+-------------+-----------+-------------------+---------+
|customer_id|campaign_id|campaign_name   |channel|send_date |campaign_month|opened|clicked|is_engaged|touch_cost_usd|segment   |region|churned_30d|arr_usd|tenure_months|tenure_band|last_login_days_ago|risk_flag|
+-----------+-----------+----------------+-------+----------+--------------+------+-------+----------+--------------+----------+------+-----------+-------+-------------+-----------+-------------------+---------+
|101        |CAMP-001   |Winback Email A |email  |2026-03-01|2026-03       |1     |0      |1         |0.12          |Smb       |NA    |1          |120.0  |3            |0-5        |14                 |0        |
|102        |CAMP-001   |Winback Email A |email  |2026-03-01|2026-03       |1     |1      |1         |0.12          |Enterprise|NA    |0          |800.0

## 4) Build dashboard-ready metrics

In [5]:
dashboard_df = (
    fact_df
    .groupBy("campaign_month", "channel", "segment", "region")
    .agg(
        F.count("*").alias("customers_reached"),
        F.sum("opened").alias("opens"),
        F.sum("clicked").alias("clicks"),
        F.sum("is_engaged").alias("engaged_customers"),
        F.sum("churned_30d").alias("churned_customers"),
        F.avg("arr_usd").alias("avg_arr_usd"),
        F.avg("touch_cost_usd").alias("avg_touch_cost_usd"),
        F.sum("touch_cost_usd").alias("total_touch_cost_usd"),
        F.sum(F.when(F.col("churned_30d") == 1, F.col("arr_usd")).otherwise(F.lit(0.0))).alias("churned_arr_usd")
    )
    .withColumn("open_rate", F.round(F.col("opens") / F.col("customers_reached"), 3))
    .withColumn("click_rate", F.round(F.col("clicks") / F.col("customers_reached"), 3))
    .withColumn("engagement_rate", F.round(F.col("engaged_customers") / F.col("customers_reached"), 3))
    .withColumn("churn_rate", F.round(F.col("churned_customers") / F.col("customers_reached"), 3))
    .withColumn("roi_proxy", F.round(F.col("churned_arr_usd") / F.when(F.col("total_touch_cost_usd") == 0, F.lit(None)).otherwise(F.col("total_touch_cost_usd")), 2))
    .orderBy("campaign_month", "channel", "segment", "region")
)

dashboard_df.show(truncate=False)

+--------------+-------+----------+------+-----------------+-----+------+-----------------+-----------------+-----------+------------------+--------------------+---------------+---------+----------+---------------+----------+---------+
|campaign_month|channel|segment   |region|customers_reached|opens|clicks|engaged_customers|churned_customers|avg_arr_usd|avg_touch_cost_usd|total_touch_cost_usd|churned_arr_usd|open_rate|click_rate|engagement_rate|churn_rate|roi_proxy|
+--------------+-------+----------+------+-----------------+-----+------+-----------------+-----------------+-----------+------------------+--------------------+---------------+---------+----------+---------------+----------+---------+
|2026-03       |call   |Enterprise|NA    |1                |0    |0     |0                |1                |1200.0     |2.5               |2.5                 |1200.0         |0.0      |0.0       |0.0            |1.0       |480.0    |
|2026-03       |call   |Smb       |EU    |1             

## 5) Simple quality checks

In [6]:
quality_checks = {
    "raw_campaign_rows": raw_campaign_df.count(),
    "raw_churn_rows": raw_churn_df.count(),
    "fact_rows_after_join": fact_df.count(),
    "null_churn_labels": fact_df.filter(F.col("churned_30d").isNull()).count(),
    "distinct_campaigns": fact_df.select("campaign_id").distinct().count(),
}

for k, v in quality_checks.items():
    print(f"{k}: {v}")

raw_campaign_rows: 10
raw_churn_rows: 10
fact_rows_after_join: 10
null_churn_labels: 0
distinct_campaigns: 5


In [8]:
# Register a temp view for dashboard SQL or BI exploration
dashboard_ready = dashboard_df
dashboard_ready.createOrReplaceTempView("vw_churn_campaign_dashboard")

spark.sql("""
SELECT campaign_month, channel, segment, region,
       customers_reached, churn_rate, open_rate, click_rate, roi_proxy
FROM vw_churn_campaign_dashboard
ORDER BY campaign_month, channel, segment, region
""").show(truncate=False)

+--------------+-------+----------+------+-----------------+----------+---------+----------+---------+
|campaign_month|channel|segment   |region|customers_reached|churn_rate|open_rate|click_rate|roi_proxy|
+--------------+-------+----------+------+-----------------+----------+---------+----------+---------+
|2026-03       |call   |Enterprise|NA    |1                |1.0       |0.0      |0.0       |480.0    |
|2026-03       |call   |Smb       |EU    |1                |1.0       |1.0      |0.0       |30.0     |
|2026-03       |email  |Consumer  |NA    |1                |0.0       |1.0      |1.0       |0.0      |
|2026-03       |email  |Enterprise|NA    |1                |0.0       |1.0      |1.0       |0.0      |
|2026-03       |email  |Smb       |EU    |1                |1.0       |0.0      |0.0       |791.67   |
|2026-03       |email  |Smb       |NA    |1                |1.0       |1.0      |0.0       |1000.0   |
|2026-03       |push   |Enterprise|APAC  |1                |0.0       |1.

## 6) Example output for dashboard use

In [9]:
# This is the dataset you would typically hand off to a BI tool or write to a curated table.
dashboard_ready = dashboard_df

dashboard_ready.limit(20).toPandas()

,campaign_month,channel,segment,region,customers_reached,opens,clicks,engaged_customers,churned_customers,avg_arr_usd,avg_touch_cost_usd,total_touch_cost_usd,churned_arr_usd,open_rate,click_rate,engagement_rate,churn_rate,roi_proxy
0,2026-03,call,Enterprise,NA,1,0,0,0,1,1200.0,2.50,2.50,1200.0,0.0,0.0,0.0,1.0,480.00
1,2026-03,call,Smb,EU,1,1,0,1,1,75.0,2.50,2.50,75.0,1.0,0.0,1.0,1.0,30.00
2,2026-03,email,Consumer,NA,1,1,1,1,0,200.0,0.10,0.10,0.0,1.0,1.0,1.0,0.0,0.00
3,2026-03,email,Enterprise,NA,1,1,1,1,0,800.0,0.12,0.12,0.0,1.0,1.0,1.0,0.0,0.00
4,2026-03,email,Smb,EU,1,0,0,0,1,95.0,0.12,0.12,95.0,0.0,0.0,0.0,1.0,791.67
5,2026-03,email,Smb,NA,1,1,0,1,1,120.0,0.12,0.12,120.0,1.0,0.0,1.0,1.0,1000.00
6,2026-03,push,Enterprise,APAC,1,1,1,1,0,900.0,0.05,0.05,0.0,1.0,1.0,1.0,0.0,0.00
7,2026-03,push,Smb,EU,1,1,0,1,0,150.0,0.05,0.05,0.0,1.0,0.0,1.0,0.0,0.00
8,2026-03,sms,Consumer,APAC,1,0,0,0,1,35.0,0.08,0.08,35.0,0.0,0.0,0.0,1.0,437.50
9,2026-03,sms,Consumer,NA,1,1,1,1,0,50.0,0.08,0.08,0.0,1.0,1.0,1.0,0.0,0.00
